# Kosovo Healthcare Access Analyzer
**Author:** Anid Mehmeti 
**Institution:** RIT Kosovo  
**Date:** March 2026

## Project Overview
This notebook analyzes healthcare facility accessibility across Kosovo's 
municipalities using open data. The core metric is healthcare facilities 
per 1,000 residents, used to identify underserved regions.

## Data Sources
- Kosovo Agency of Statistics — municipal population data
- OpenStreetMap (Overpass Turbo) — healthcare facility locations
- GADM — Kosovo municipality shapefiles

In [2]:
# Standard data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Geospatial
import geopandas as gpd

# Utilities
import requests
import os

print("All libraries imported successfully.")

All libraries imported successfully.


## Step 2 - Data Collection
Data is collected from three sources: KAS (population), OpenStreetMap (facilities), and GADM (shapefiles).

In [3]:
# Load population data
# Skipping first 2 rows (title + blank), using only the first 2 columns
pop_df = pd.read_excel(
    '../data/raw/population_2024.xlsx',
    header=None,
    skiprows=3,
    usecols=[0, 1],
    names=['municipality', 'population']
)

# Drop the Kosovo total row and any rows with missing values
pop_df = pop_df.dropna()
pop_df = pop_df[pop_df['municipality'] != 'Kosova']

# Reset index cleanly
pop_df = pop_df.reset_index(drop=True)

print(f"Municipalities loaded: {len(pop_df)}")
print(pop_df.head(10))

Municipalities loaded: 38
   municipality  population
0         Deçan     27429.0
1       Gjakovë     77449.0
2       Gllogoc     47121.0
3        Gjilan     81842.0
4       Dragash     28637.0
5         Istog     32464.0
6       Kaçanik     27489.0
7         Klinë     29661.0
8  Fushë Kosovë     64393.0
9      Kamenicë     22312.0


### Healthcare Facility Data
- Source: OpenStreetMap via Overpass API
- Query: All nodes/ways tagged as healthcare facilities within Kosovo
- Returns: facility name, type, and coordinates (lat/lon)

In [4]:
# Query Overpass API for healthcare facilities in Kosovo
# amenity=clinic, hospital, doctors, pharmacy, dentist covers the main types

overpass_url = "http://overpass-api.de/api/interpreter"

overpass_query = """
[out:json][timeout:60];
area["name:en"="Kosovo"]->.searchArea;
(
  node["amenity"="hospital"](area.searchArea);
  node["amenity"="clinic"](area.searchArea);
  node["amenity"="doctors"](area.searchArea);
  node["amenity"="pharmacy"](area.searchArea);
  node["amenity"="dentist"](area.searchArea);
);
out body;
"""

response = requests.get(overpass_url, params={"data": overpass_query})
data = response.json()

print(f"Status code: {response.status_code}")
print(f"Facilities found: {len(data['elements'])}")

Status code: 200
Facilities found: 822


In [5]:
# Parse the API response into a DataFrame
# Each element in data['elements'] is a facility with tags and coordinates

facilities = []

for element in data['elements']:
    facilities.append({
        'facility_id': element.get('id'),
        'amenity_type': element.get('tags', {}).get('amenity'),
        'name': element.get('tags', {}).get('name', 'Unknown'),
        'lat': element.get('lat'),
        'lon': element.get('lon')
    })

facilities_df = pd.DataFrame(facilities)

print(f"Total facilities: {len(facilities_df)}")
print(f"\nFacility types breakdown:")
print(facilities_df['amenity_type'].value_counts())
print(f"\nSample rows:")
print(facilities_df.head())

Total facilities: 822

Facility types breakdown:
amenity_type
pharmacy    351
clinic      168
dentist     141
hospital     81
doctors      81
Name: count, dtype: int64

Sample rows:
   facility_id amenity_type                name        lat        lon
0    454697962     pharmacy              Humana  42.575866  21.579319
1    454704042     hospital       QKMF Kamenica  42.576347  21.580115
2    517589630     hospital  Klinika Kirurgjike  42.644364  21.162371
3    603647263      doctors            Ars Dent  42.668393  21.175257
4    612468828      doctors           Kirurgjia  42.657581  21.156246


### Data Decision: Healthcare Facility Types
**Included in primary care metric:** hospitals, clinics, doctors  
**Excluded:** pharmacies, dentists  

**Rationale:** Pharmacies dispense medication but do not provide diagnosis or 
treatment — a municipality with only pharmacies is still medically underserved. 
Dentists are excluded as they represent specialized rather than primary care.

**Note:** Pharmacy data is retained separately for potential secondary analysis 
(pharmacy-to-doctor ratio in underserved regions).

In [6]:

# Separate primary care facilities from the full dataset
primary_care_df = facilities_df[
    facilities_df['amenity_type'].isin(['hospital', 'clinic', 'doctors'])
].copy()

# Keep pharmacies separately for potential secondary analysis
pharmacy_df = facilities_df[
    facilities_df['amenity_type'] == 'pharmacy'
].copy()

print(f"Primary care facilities: {len(primary_care_df)}")
print(f"Pharmacies retained separately: {len(pharmacy_df)}")
print(f"\nBreakdown:")
print(primary_care_df['amenity_type'].value_counts())

Primary care facilities: 330
Pharmacies retained separately: 351

Breakdown:
amenity_type
clinic      168
hospital     81
doctors      81
Name: count, dtype: int64


In [7]:
# Save raw facility data to disk so we don't need to re-query the API
facilities_df.to_csv('../data/raw/facilities_all.csv', index=False)
primary_care_df.to_csv('../data/raw/facilities_primary_care.csv', index=False)
pharmacy_df.to_csv('../data/raw/facilities_pharmacy.csv', index=False)

print("Facility data saved to data/raw/")

Facility data saved to data/raw/


In [8]:
# Load Kosovo municipality boundaries
kosovo_map = gpd.read_file('../data/raw/gadm41_XKO_2.shp')

print(f"Shape of geodataframe: {kosovo_map.shape}")
print(f"\nColumns: {kosovo_map.columns.tolist()}")
print(f"\nSample municipality names:")
print(kosovo_map['NAME_2'].to_string())

Shape of geodataframe: (30, 14)

Columns: ['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'geometry']

Sample municipality names:
0               Đakovica
1                 Dečani
2               Orahovac
3               Gnjilane
4      Kosovska Kamenica
5                 Vitina
6     Kosovska Mitrovica
7              Leposavić
8                 Srbica
9                Vucitrn
10           Zubin Potok
11                Zvečan
12                 Istok
13                 Klina
14                   Peć
15              Glogovac
16          Kosovo Polje
17               Lipljan
18             Novo Brdo
19                Obilić
20              Podujevo
21              Priština
22                Dragaš
23              Mališevo
24               Prizren
25             Suva Reka
26               Kačanik
27               Štimlje
28                Štrpce
29              Uroševac


In [9]:
# Translate Serbian municipality names to Albanian
name_translation = {
    'Đakovica': 'Gjakovë',
    'Dečani': 'Deçan',
    'Orahovac': 'Rahovec',
    'Gnjilane': 'Gjilan',
    'Kosovska Kamenica': 'Kamenicë',
    'Vitina': 'Viti',
    'Kosovska Mitrovica': 'Mitrovicë',
    'Leposavić': 'Leposaviq',
    'Srbica': 'Sknderaj',
    'Vucitrn': 'Vushtrri',
    'Zubin Potok': 'Zubin Potok',
    'Zvečan': 'Zveçan',
    'Istok': 'Istog',
    'Klina': 'Klinë',
    'Peć': 'Pejë',
    'Glogovac': 'Gllogoc',
    'Kosovo Polje': 'Fushë Kosovë',
    'Lipljan': 'Lipjan',
    'Novo Brdo': 'Novobërdë',
    'Obilić': 'Obiliq',
    'Podujevo': 'Podujevë',
    'Priština': 'Prishtinë',
    'Dragaš': 'Dragash',
    'Mališevo': 'Malishevë',
    'Prizren': 'Prizren',
    'Suva Reka': 'Suharekë',
    'Kačanik': 'Kaçanik',
    'Štimlje': 'Shtime',
    'Štrpce': 'Shtërpcë',
    'Uroševac': 'Ferizaj'
}

# Apply translation
kosovo_map['municipality'] = kosovo_map['NAME_2'].map(name_translation)

# Now compare with population data to find missing municipalities
gadm_names = set(kosovo_map['municipality'])
kas_names = set(pop_df['municipality'])

missing_from_gadm = kas_names - gadm_names
missing_from_kas = gadm_names - kas_names

print("In KAS but missing from GADM shapefile:")
for name in sorted(missing_from_gadm):
    print(f"  - {name}")


In KAS but missing from GADM shapefile:
  - Graçanicë
  - Hani i Elezit
  - Junik
  - Kllokot
  - Mamushë
  - Mitrovica e V.
  - Partesh
  - Ranillug


In [10]:
missing = ['Graçanicë', 'Hani i Elezit', 'Junik', 'Kllokot', 
           'Mamushë', 'Mitrovica e V.', 'Partesh', 'Ranillug']

print(pop_df[pop_df['municipality'].isin(missing)][['municipality', 'population']])
print(f"\nTotal population in missing municipalities: {pop_df[pop_df['municipality'].isin(missing)]['population'].sum():.0f}")
print(f"As % of Kosovo total: {pop_df[pop_df['municipality'].isin(missing)]['population'].sum() / pop_df['population'].sum() * 100:.1f}%")

      municipality  population
30           Junik      3905.0
31         Mamushë      5636.0
32   Hani i Elezit      8555.0
33       Graçanicë     18592.0
34        Ranillug      2472.0
35         Partesh      3226.0
36         Kllokot      3019.0
37  Mitrovica e V.      7923.0

Total population in missing municipalities: 53328
As % of Kosovo total: 3.4%


### Data Decision: Excluded Municipalities
The following 8 municipalities are excluded from analysis due to missing 
GADM shapefile boundaries: Graçanicë, Hani i Elezit, Junik, Kllokot, 
Mamushë, Mitrovica e V., Partesh, Ranillug.

These represent 3.4% of Kosovo's population (53,328 residents).

**Rationale for exclusion rather than reassignment:**
Several of these are predominantly Serb municipalities (Ranillug, Partesh, 
Kllokot, Graçanicë, Mitrovica e V.) where residents may seek healthcare 
outside Kosovo's system entirely — in Serbia or through parallel institutions. 
Reassigning their population to Albanian-majority municipalities would 
misrepresent both the geographic and political reality on the ground.
This is a known limitation of the analysis and should be considered by 
any policymaker reading these findings.

In [11]:
# Keep only municipalities present in both datasets
valid_municipalities = set(kosovo_map['municipality'])

pop_df_filtered = pop_df[
    pop_df['municipality'].isin(valid_municipalities)
].reset_index(drop=True)

print(f"Municipalities retained: {len(pop_df_filtered)}")
print(f"Municipalities excluded: {38 - len(pop_df_filtered)}")
print(f"\nPopulation covered: {pop_df_filtered['population'].sum():,.0f}")
print(f"As % of Kosovo total: {pop_df_filtered['population'].sum() / pop_df['population'].sum() * 100:.1f}%")

Municipalities retained: 30
Municipalities excluded: 8

Population covered: 1,532,262
As % of Kosovo total: 96.6%


In [12]:
# Save all processed data from Step 2
pop_df_filtered.to_csv('../data/processed/population_filtered.csv', index=False)
kosovo_map.to_file('../data/processed/kosovo_municipalities.gpkg', driver='GPKG')

print("Saved:")
print("  - data/processed/population_filtered.csv")
print("  - data/processed/kosovo_municipalities.gpkg")

Saved:
  - data/processed/population_filtered.csv
  - data/processed/kosovo_municipalities.gpkg


## Step 3 - Data Cleaning and Merging
Facilities are assigned to municipalities via spatial join using coordinates.
Final DataFrame contains population, facility count, and facilities-per-1000 ratio.

In [13]:
# Reload processed data
pop_df_filtered = pd.read_csv('../data/processed/population_filtered.csv')
kosovo_map = gpd.read_file('../data/processed/kosovo_municipalities.gpkg')
primary_care_df = pd.read_csv('../data/raw/facilities_primary_care.csv')

print(f"Population data: {len(pop_df_filtered)} municipalities")
print(f"Shapefile: {len(kosovo_map)} municipalities")
print(f"Facilities: {len(primary_care_df)} primary care facilities")

Population data: 30 municipalities
Shapefile: 30 municipalities
Facilities: 330 primary care facilities


In [14]:
# Convert facilities DataFrame to a GeoDataFrame
# This turns lat/lon columns into actual geometry points
facilities_gdf = gpd.GeoDataFrame(
    primary_care_df,
    geometry=gpd.points_from_xy(primary_care_df['lon'], primary_care_df['lat']),
    crs='EPSG:4326'  # Standard GPS coordinate system
)

# Make sure both layers use the same coordinate system
kosovo_map = kosovo_map.to_crs('EPSG:4326')

# Spatial join — assigns each facility to the municipality it falls within
facilities_joined = gpd.sjoin(
    facilities_gdf,
    kosovo_map[['municipality', 'geometry']],
    how='left',
    predicate='within'
)

print(f"Total facilities: {len(facilities_joined)}")
print(f"Facilities successfully assigned to a municipality: {facilities_joined['municipality'].notna().sum()}")
print(f"Facilities not assigned: {facilities_joined['municipality'].isna().sum()}")

Total facilities: 330
Facilities successfully assigned to a municipality: 329
Facilities not assigned: 1


In [15]:
# Drop the one unassigned facility
facilities_joined = facilities_joined.dropna(subset=['municipality'])

print(f"Facilities retained: {len(facilities_joined)}")

# Count facilities per municipality
facility_counts = facilities_joined.groupby('municipality').size().reset_index(name='facility_count')

print(f"\nMunicipalities with at least one facility: {len(facility_counts)}")
print(f"\nFacility counts sample:")
print(facility_counts.sort_values('facility_count', ascending=False).head(10))

Facilities retained: 329

Municipalities with at least one facility: 27

Facility counts sample:
   municipality  facility_count
18    Prishtinë             130
4       Gjakovë              38
19      Prizren              28
5        Gjilan              23
16         Pejë              20
9       Kaçanik              13
12       Lipjan              10
25     Vushtrri              10
23     Suharekë               8
2       Ferizaj               7


In [16]:
# Check Mitrovicë specifically
print("Mitrovicë in facility counts:")
print(facility_counts[facility_counts['municipality'] == 'Mitrovicë'])

# Find which municipalities have zero facilities
all_municipalities = set(pop_df_filtered['municipality'])
municipalities_with_facilities = set(facility_counts['municipality'])
zero_facilities = all_municipalities - municipalities_with_facilities

print(f"\nMunicipalities with zero facilities assigned: {len(zero_facilities)}")
for m in sorted(zero_facilities):
    print(f"  - {m}")

Mitrovicë in facility counts:
   municipality  facility_count
14    Mitrovicë               4

Municipalities with zero facilities assigned: 3
  - Novobërdë
  - Shtërpcë
  - Zubin Potok


In [18]:
# Full facility count breakdown sorted descending
print("All municipalities by facility count:")
print(facility_counts.sort_values('facility_count', ascending=False).to_string())

print("\nFacilities assigned to Mitrovicë:")
print(facilities_joined[facilities_joined['municipality'] == 'Mitrovicë'][['name', 'amenity_type', 'lat', 'lon']])

print("\nFacilities assigned to Prishtinë:")
print(facilities_joined[facilities_joined['municipality'] == 'Prishtinë'][['name', 'amenity_type', 'lat', 'lon']])

All municipalities by facility count:
    municipality  facility_count
18     Prishtinë             130
4        Gjakovë              38
19       Prizren              28
5         Gjilan              23
16          Pejë              20
9        Kaçanik              13
12        Lipjan              10
25      Vushtrri              10
23      Suharekë               8
2        Ferizaj               7
6        Gllogoc               5
7          Istog               4
3   Fushë Kosovë               4
14     Mitrovicë               4
1        Dragash               3
22      Sknderaj               3
17      Podujevë               3
0          Deçan               3
8       Kamenicë               2
10         Klinë               2
21        Shtime               2
13     Malishevë               2
11     Leposaviq               1
15        Obiliq               1
20       Rahovec               1
24          Viti               1
26        Zveçan               1

Facilities assigned to Mitrovicë:
   

In [19]:
# Find municipalities with zero facilities
all_municipalities = set(pop_df_filtered['municipality'])
municipalities_with_facilities = set(facility_counts['municipality'])
zero_facilities = all_municipalities - municipalities_with_facilities

print("Municipalities with zero recorded facilities:")
for m in sorted(zero_facilities):
    pop = pop_df_filtered[pop_df_filtered['municipality'] == m]['population'].values[0]
    print(f"  - {m}: population {pop:,.0f}")

Municipalities with zero recorded facilities:
  - Novobërdë: population 4,438
  - Shtërpcë: population 10,774
  - Zubin Potok: population 3,381


In [20]:
# Check ALL facility types for these municipalities (not just primary care)
all_facilities_gdf = gpd.GeoDataFrame(
    facilities_df,
    geometry=gpd.points_from_xy(facilities_df['lon'], facilities_df['lat']),
    crs='EPSG:4326'
)

all_joined = gpd.sjoin(
    all_facilities_gdf,
    kosovo_map[['municipality', 'geometry']],
    how='left',
    predicate='within'
)

zero_munis = ['Novobërdë', 'Shtërpcë', 'Zubin Potok']
for m in zero_munis:
    count = all_joined[all_joined['municipality'] == m]
    print(f"\n{m} — all facility types:")
    if len(count) == 0:
        print("  No facilities of any type recorded in OSM")
    else:
        print(count[['name', 'amenity_type']])


Novobërdë — all facility types:
  No facilities of any type recorded in OSM

Shtërpcë — all facility types:
  No facilities of any type recorded in OSM

Zubin Potok — all facility types:
  No facilities of any type recorded in OSM


In [21]:
missing_facilities = ['Novobërdë', 'Shtërpcë', 'Zubin Potok']

print(pop_df_filtered[
    pop_df_filtered['municipality'].isin(missing_facilities)
][['municipality', 'population']])

   municipality  population
13    Novobërdë      4438.0
22     Shtërpcë     10774.0
27  Zubin Potok      3381.0


### Data Decision: Municipalities with Zero OSM Facilities
Novobërdë, Shtërpcë, and Zubin Potok show zero facilities in OpenStreetMap.
These are predominantly Serb municipalities where OSM coverage is known to be 
incomplete. A zero here reflects a data gap, not reality.

Combined with the 8 previously excluded municipalities, all 11 excluded 
municipalities are predominantly Serb. This is a significant limitation 
that any policymaker should be aware of when reading this analysis.

These 3 municipalities are excluded from calculations. Final analysis 
covers 27 municipalities.

In [22]:
# Remove municipalities with zero OSM facilities
zero_facility_municipalities = ['Novobërdë', 'Shtërpcë', 'Zubin Potok']

pop_df_clean = pop_df_filtered[
    ~pop_df_filtered['municipality'].isin(zero_facility_municipalities)
].reset_index(drop=True)

kosovo_map_clean = kosovo_map[
    ~kosovo_map['municipality'].isin(zero_facility_municipalities)
].reset_index(drop=True)

print(f"Municipalities in final analysis: {len(pop_df_clean)}")
print(f"Population covered: {pop_df_clean['population'].sum():,.0f}")
print(f"As % of Kosovo total: {pop_df_clean['population'].sum() / pop_df['population'].sum() * 100:.1f}%")

Municipalities in final analysis: 27
Population covered: 1,513,669
As % of Kosovo total: 95.5%


In [23]:
# Merge population and facility counts into one DataFrame
merged_df = pop_df_clean.merge(facility_counts, on='municipality', how='left')

# Fill any remaining zeros (shouldn't be any, but just in case)
merged_df['facility_count'] = merged_df['facility_count'].fillna(0).astype(int)

# Calculate the core metric: facilities per 1000 residents
merged_df['facilities_per_1000'] = (
    merged_df['facility_count'] / merged_df['population'] * 1000
).round(3)

# Sort by facilities_per_1000 for easy reading
merged_df = merged_df.sort_values('facilities_per_1000', ascending=False).reset_index(drop=True)

print("Final merged DataFrame:")
print(merged_df.to_string())

Final merged DataFrame:
    municipality  population  facility_count  facilities_per_1000
0      Prishtinë    227737.0             130                0.571
1        Gjakovë     77449.0              38                0.491
2        Kaçanik     27489.0              13                0.473
3         Zveçan      2859.0               1                0.350
4         Gjilan     81842.0              23                0.281
5           Pejë     81751.0              20                0.245
6        Prizren    145938.0              28                0.192
7         Lipjan     54571.0              10                0.183
8       Suharekë     44690.0               8                0.179
9       Vushtrri     60658.0              10                0.165
10         Istog     32464.0               4                0.123
11         Deçan     27429.0               3                0.109
12       Gllogoc     47121.0               5                0.106
13       Dragash     28637.0               3        